In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import copy
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, TensorDataset


# CONFIG

DATA_PATH   = "/kaggle/input/datasets/arjunmahesh09999/new-masterdata/MASTERDATA.csv"
OUTPUT_DIR  = Path("cnn_gru_v6_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED     = 7
ROWS_PER_MIN    = 30

BURN_IN_ROWS    = 320
TRIM_FIRST_ROWS = 200
TRIM_LAST_ROWS  = 200

# Val/test get a stricter first-row trim
VAL_TEST_TRIM_FIRST = 400
VAL_TEST_TRIM_LAST  = 200

WINDOW_ROWS = 8 * ROWS_PER_MIN   # 240 rows
STRIDE_ROWS = 20

TEST_PATIENT_FRAC = 0.10
VAL_PATIENT_FRAC  = 0.10

TARGET_COL  = "future_label"
PATIENT_COL = "patient_id"

BATCH_SIZE   = 256
MAX_EPOCHS   = 60          # ↑ from 50 — deeper model may need more epochs
LR           = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 20
LABEL_SMOOTH = 0.02
GRAD_CLIP    = 5.0


CLASS1_BOOST = 1.5   # critical  (was 1.3 in v5)
CLASS2_BOOST = 1.2   # emergency (NEW — was 1.0 before)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================
# FEATURES  
# ============================================================
VITAL_LIMITS = {
    "dbp":               (20, 150),
    "mbp":               (30, 200),
    "sbp":               (50, 280),
    "heart_rate":        (20, 220),
    "spo2":              (50, 100),
    "etco2":             (5,  80),
    "pulse_pressure":    (5,  150),
    "resp_rate_smoothed":(4,  60),
}
RAW_VITALS = list(VITAL_LIMITS.keys())

SLOPE_PREFS  = ["slope_5m_", "slope_7m_", "slope_15m_"]
SLOPE_VITALS = ["spo2","heart_rate","resp_rate_smoothed","sbp","dbp","mbp","etco2","pulse_pressure"]
SLOPE_COLS   = [f"{p}{v}" for p in SLOPE_PREFS for v in SLOPE_VITALS]

COMBINED_SLOPE_COLS = [
    "slope_2m_combined_score",
    "slope_5m_combined_score",
    "slope_7m_combined_score",
    "slope_15m_combined_score",
]

EXTRA_COLS = [
    "combined_score",
    "roll_mean_15m_combined",
    "roll_min_15m_combined",
    "roll_max_15m_combined",
    "roll_std_15m_combined",
]

TARGETED_FLAG_COLS = [
    "t3_masked_shock",
    "t3_stable_deceiver",
    "t3_occult_acidosis",
]

VITAL_NORM = {
    "dbp":               (70.0,  12.5),
    "mbp":               (90.0,  15.0),
    "sbp":              (120.0,  23.75),
    "heart_rate":        (75.0,  18.75),
    "spo2":              (98.0,   2.5),
    "etco2":             (40.0,   7.5),
    "pulse_pressure":    (50.0,  13.75),
    "resp_rate_smoothed":(16.0,   5.5),
    "combined_score":           (0.0,  0.5),
    "roll_mean_15m_combined":   (0.2,  0.4),
    "roll_min_15m_combined":    (0.0,  0.4),
    "roll_max_15m_combined":    (0.3,  0.4),
    "roll_std_15m_combined":    (0.0,  0.15),
}

SLOPE_VITAL_STD = {
    "slope_5m":  0.015,
    "slope_7m":  0.012,
    "slope_15m": 0.008,
}
COMBINED_SLOPE_STD = {
    "slope_2m_combined_score": 0.005,
    "slope_5m_combined_score": 0.003,
    "slope_7m_combined_score": 0.002,
    "slope_15m_combined_score":0.001,
}

# ============================================================
# PREPROCESSING
# ============================================================
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    print("* Feature engineering...")

    def _patient(g):
        g = g.copy()
        for v in RAW_VITALS:
            if v not in g.columns:
                continue
            s    = g[v].astype(float)
            prev = s.shift(1)
            jump = (prev.notna()) & ((s > 2.0*prev.abs()) | (s < 0.5*prev.abs()))
            s[jump] = np.nan
            g[v] = s.ffill()
        for v, (lo, hi) in VITAL_LIMITS.items():
            if v in g.columns:
                g.loc[(g[v] < lo) | (g[v] > hi), v] = np.nan
        return g

    df = df.groupby(PATIENT_COL, group_keys=False).apply(_patient).reset_index(drop=True)
    print(f"  Columns after engineering: {df.shape[1]}")
    return df


def trim_edges(df: pd.DataFrame, first_rows: int = None, last_rows: int = None) -> pd.DataFrame:
    if first_rows is None:
        first_rows = BURN_IN_ROWS + TRIM_FIRST_ROWS
    if last_rows is None:
        last_rows = TRIM_LAST_ROWS

    print(f"* Trimming first {first_rows} and last {last_rows} rows...")

    def _trim(g):
        if len(g) <= (first_rows + last_rows):
            return g.iloc[0:0]
        return g.iloc[first_rows : len(g) - last_rows]

    return df.groupby(PATIENT_COL, group_keys=False).apply(_trim).reset_index(drop=True)


def build_feature_cols(df: pd.DataFrame):
    candidates = (
        RAW_VITALS + SLOPE_COLS + COMBINED_SLOPE_COLS +
        EXTRA_COLS + TARGETED_FLAG_COLS
    )
    present = [c for c in candidates if c in df.columns]
    print(f"* Final Feature Count: {len(present)}")
    return present


def apply_fixed_normalisation(X, feature_cols):
    X = np.nan_to_num(X.copy().astype(np.float32), nan=0.0)
    for fi, col in enumerate(feature_cols):
        if col in VITAL_NORM:
            ref, std = VITAL_NORM[col]
            X[:, :, fi] = (X[:, :, fi] - ref) / (std + 1e-8)
            continue
        matched = False
        for key, std in COMBINED_SLOPE_STD.items():
            if key in col:
                X[:, :, fi] /= (std + 1e-8)
                matched = True
                break
        if matched:
            continue
        for prefix, std in SLOPE_VITAL_STD.items():
            if prefix in col:
                X[:, :, fi] /= (std + 1e-8)
                break
    return X


# ============================================================
# SPLIT + WINDOWS
# ============================================================
def split_patients(df: pd.DataFrame):
    pids = df[PATIENT_COL].unique().copy()
    rng  = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(pids)

    n_test     = max(1, int(len(pids) * TEST_PATIENT_FRAC))
    test_pids  = pids[:n_test]
    rem        = pids[n_test:]
    n_val      = max(1, int(len(rem) * VAL_PATIENT_FRAC))
    val_pids   = rem[:n_val]
    train_pids = rem[n_val:]

    train_df = df[df[PATIENT_COL].isin(train_pids)].reset_index(drop=True)
    val_df   = df[df[PATIENT_COL].isin(val_pids)].reset_index(drop=True)
    test_df  = df[df[PATIENT_COL].isin(test_pids)].reset_index(drop=True)

    print(f"* Splits (before trim): Train={len(train_pids)}, Val={len(val_pids)}, Test={len(test_pids)}")
    return train_df, val_df, test_df


def make_windows(df: pd.DataFrame, feature_cols):
    X_list, y_list, pid_list = [], [], []
    for pid in df[PATIENT_COL].unique():
        g      = df[df[PATIENT_COL] == pid].reset_index(drop=True)
        feat   = g[feature_cols].values.astype(np.float32)
        labels = g[TARGET_COL].values
        n      = len(g)
        start  = 0
        while start + WINDOW_ROWS <= n:
            end   = start + WINDOW_ROWS
            label = labels[end - 1]
            if not np.isnan(label):
                X_list.append(feat[start:end])
                y_list.append(int(label))
                pid_list.append(pid)
            start += STRIDE_ROWS

    if len(X_list) == 0:
        return (
            np.empty((0, WINDOW_ROWS, len(feature_cols)), dtype=np.float32),
            np.array([]), np.array([]),
        )
    return np.stack(X_list), np.array(y_list), np.array(pid_list)


# ============================================================
# MODEL  — deeper GRU (2 layers) to handle larger dataset
# ============================================================
class CNNGRU(nn.Module):
    
    def __init__(self, n_features, n_classes=3, dropout=0.4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(n_features, 48, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(48), nn.GELU(),
            nn.Conv1d(48, 72, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(72), nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(72, 72, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(72), nn.GELU(),
        )
        # ── 2-layer bidirectional GRU ─────────────────────────────────────
        self.gru = nn.GRU(
            input_size=72, hidden_size=48,
            num_layers=2,           
            batch_first=True,
            bidirectional=True,
            dropout=dropout,        
        )
        self.attn = nn.Sequential(nn.Linear(96, 48), nn.Tanh(), nn.Linear(48, 1))
        self.head = nn.Sequential(
            nn.Linear(96, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, x):
        z    = self.cnn(x).transpose(1, 2)      # (B, T, 72)
        h, _ = self.gru(z)                       # (B, T, 96)
        w    = torch.softmax(self.attn(h).squeeze(-1), dim=1)
        ctx  = (h * w.unsqueeze(-1)).sum(dim=1)  # (B, 96)
        return self.head(ctx)                    # (B, 3)


def make_loader(X, y, shuffle):
    Xt = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(
        TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=2, pin_memory=(DEVICE.type == "cuda"),
    )


# ============================================================
# TEMPERATURE SCALING  ← NEW
# ============================================================
class TemperatureScaler(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def calibrate(self, logits_np, labels_np):
        """Fit temperature on validation logits."""
        logits_t = torch.tensor(logits_np, dtype=torch.float32).to(DEVICE)
        labels_t = torch.tensor(labels_np, dtype=torch.long).to(DEVICE)

        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def _eval():
            optimizer.zero_grad()
            loss = F.cross_entropy(logits_t / self.temperature, labels_t)
            loss.backward()
            return loss

        optimizer.step(_eval)
        T = self.temperature.item()
        print(f"  Learned temperature T = {T:.4f}  (>1 = softer, <1 = sharper)")
        return T


# ============================================================
# DUAL-CLASS THRESHOLD TUNING  ← NEW
# ============================================================
def tune_dual_thresholds(y_true, proba, t1_range=None, t2_range=None):
    
    if t1_range is None:
        t1_range = np.arange(0.22, 0.42, 0.02)
    if t2_range is None:
        t2_range = np.arange(0.28, 0.52, 0.02)

    best_score        = -1.0
    best_t1, best_t2  = 0.33, 0.40
    results           = []

    for t1 in t1_range:
        for t2 in t2_range:
            preds = _apply_dual_threshold(proba, t1, t2)
            f1_c1 = f1_score(y_true, preds, labels=[1], average="macro", zero_division=0)
            f1_c2 = f1_score(y_true, preds, labels=[2], average="macro", zero_division=0)
            rec1  = recall_score(y_true, preds, labels=[1], average="macro", zero_division=0)
            rec2  = recall_score(y_true, preds, labels=[2], average="macro", zero_division=0)
            bal   = balanced_accuracy_score(y_true, preds)
            score = (f1_c1 + f1_c2) / 2.0   # optimise both jointly

            results.append({
                "t1": round(t1, 2), "t2": round(t2, 2),
                "f1_crit": f1_c1, "f1_emerg": f1_c2,
                "rec_crit": rec1, "rec_emerg": rec2,
                "bal_acc": bal, "joint_score": score,
            })
            if score > best_score:
                best_score = score
                best_t1, best_t2 = t1, t2

    df_r = pd.DataFrame(results)

    # Print only top-20 rows by joint_score for readability
    top = df_r.nlargest(20, "joint_score")
    print("\n── Dual threshold sweep — top 20 (val set) ──")
    print(top.to_string(index=False, float_format="%.3f"))
    print(f"\n  → Best: t1(critical)={best_t1:.2f}, t2(emergency)={best_t2:.2f}"
          f"  (joint F1={best_score:.3f})")
    return best_t1, best_t2, df_r


def _apply_dual_threshold(proba, t1, t2):
    """
    Priority: Critical (class 1) > Emergency (class 2) > Normal (class 0)
    """
    preds = np.full(len(proba), 0, dtype=int)         # default: Normal
    preds[proba[:, 2] >= t2] = 2                       # Emergency
    preds[proba[:, 1] >= t1] = 1                       # Critical overrides Emergency
    return preds


# ============================================================
# COLLECT LOGITS (needed for temperature scaling)
# ============================================================
@torch.no_grad()
def collect_logits(model, loader):
    """Return raw logits (pre-softmax) for the entire loader."""
    model.eval()
    all_logits, all_y = [], []
    for X_b, y_b in loader:
        X_b = X_b.to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(X_b)
        all_logits.append(logits.float().cpu().numpy())
        all_y.append(y_b.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)


# ============================================================
# TRAIN / EVAL
# ============================================================
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = correct = n = 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        n          += len(y_b)
    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, temp_scaler=None):
    model.eval()
    total_loss = correct = n = 0
    all_probs, all_y = [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(X_b)
            if temp_scaler is not None:
                logits = temp_scaler(logits)
            loss = criterion(logits, y_b)
        probs       = torch.softmax(logits.float(), dim=1)
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        n          += len(y_b)
        all_probs.append(probs.cpu().numpy())
        all_y.append(y_b.cpu().numpy())
    proba = np.concatenate(all_probs)
    y     = np.concatenate(all_y)
    try:
        auroc = roc_auc_score(y, proba, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")
    return total_loss / n, correct / n, auroc, y, proba


# ============================================================
# FIT
# ============================================================
def fit_model(model, tr_loader, va_loader, class_weights):
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE),
        label_smoothing=LABEL_SMOOTH,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=LR * 0.05
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_auroc = -1.0
    best_state = copy.deepcopy(model.state_dict())
    patience   = 0
    history    = {"train_loss":[], "val_loss":[], "val_auroc":[]}

    print(f"\n{'Ep':>4} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} "
          f"{'Val Acc':>9} {'Val AUROC':>10}")
    print("-"*62)

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss, tr_acc                  = train_epoch(model, tr_loader, optimizer, criterion, scaler)
        va_loss, va_acc, va_auroc, _, _  = eval_epoch(model, va_loader, criterion)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["val_auroc"].append(va_auroc)

        mark = " *" if va_auroc > best_auroc else ""
        print(f"{ep:4d} {tr_loss:11.4f} {tr_acc:10.4f} {va_loss:10.4f} "
              f"{va_acc:9.4f} {va_auroc:10.4f}{mark}")

        if va_auroc > best_auroc:
            best_auroc = va_auroc
            best_state = copy.deepcopy(model.state_dict())
            patience   = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print(f"\n* Early stopping at epoch {ep}")
                break

    model.load_state_dict(best_state)
    return model, history, best_auroc


# ============================================================
# REPORTING
# ============================================================
def decision_report(y_true, proba, label, t1=None, t2=None):
    if t1 is None and t2 is None:
        y_pred = proba.argmax(axis=1)
    else:
        y_pred = _apply_dual_threshold(proba, t1, t2)
        print(f"  (t_critical={t1:.2f}, t_emergency={t2:.2f})")

    y_bin  = label_binarize(y_true, classes=[0, 1, 2])
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    print(classification_report(y_true, y_pred,
                                 target_names=["Normal","Critical","Emergency"]))

    auroc = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
    auprc = np.mean([average_precision_score(y_bin[:, i], proba[:, i]) for i in range(3)])
    bal   = balanced_accuracy_score(y_true, y_pred)
    print(f"  AUROC={auroc:.4f}  AUPRC={auprc:.4f}  BAL_ACC={bal:.4f}")
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    return {"auroc":auroc,"auprc":auprc,"bal":bal,
            "y_true":y_true,"y_pred":y_pred,"proba":proba,"cm":cm}


# ============================================================
# PLOTTING
# ============================================================
def plot_training(history, out_path):
    fig, ax = plt.subplots(figsize=(8, 5), facecolor="white")
    ax.plot(history["train_loss"], label="train_loss")
    ax.plot(history["val_loss"],   label="val_loss")
    ax2 = ax.twinx()
    ax2.plot(history["val_auroc"], color="C2", label="val_auroc")
    ax.set_title("Training curves"); ax.set_xlabel("Epoch")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    fig.tight_layout(); fig.savefig(out_path, dpi=140); plt.close(fig)


def plot_confusion(cm, out_path, title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks([0,1,2]); ax.set_xticklabels(["Normal","Critical","Emergency"])
    ax.set_yticks([0,1,2]); ax.set_yticklabels(["Normal","Critical","Emergency"])
    fig.colorbar(im, ax=ax)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.tight_layout(); fig.savefig(out_path, dpi=140); plt.close(fig)


def plot_threshold_heatmap(df_r, out_path):
    """Plot joint-F1 as a heatmap over (t1, t2) grid."""
    pivot = df_r.pivot_table(index="t1", columns="t2", values="joint_score")
    fig, ax = plt.subplots(figsize=(9, 6), facecolor="white")
    im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto",
                   origin="lower", vmin=0, vmax=pivot.values.max())
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{v:.2f}" for v in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.2f}" for v in pivot.index])
    ax.set_xlabel("t2 (emergency threshold)")
    ax.set_ylabel("t1 (critical threshold)")
    ax.set_title("Joint F1(critical+emergency) — val set")
    fig.colorbar(im, ax=ax)
    fig.tight_layout(); fig.savefig(out_path, dpi=140); plt.close(fig)


# ============================================================
# MAIN
# ============================================================
def main():
    
    print(" CNN-GRU")
   

    df = pd.read_csv(DATA_PATH)
    print(f"* Initial Data Shape: {df.shape}")

    df = preprocess(df)
    df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
    df[TARGET_COL] = df[TARGET_COL].astype(int)

    FEATURES = build_feature_cols(df)

    # Split first, then trim per split
    train_df, val_df, test_df = split_patients(df)

    print("\n* Applying per-split trimming...")
    train_df = trim_edges(train_df)
    val_df   = trim_edges(val_df,  VAL_TEST_TRIM_FIRST, VAL_TEST_TRIM_LAST)
    test_df  = trim_edges(test_df, VAL_TEST_TRIM_FIRST, VAL_TEST_TRIM_LAST)

    print("\n* Building windows...")
    X_tr, y_tr, _ = make_windows(train_df, FEATURES)
    X_va, y_va, _ = make_windows(val_df,   FEATURES)
    X_te, y_te, _ = make_windows(test_df,  FEATURES)

    print(f"  Train: {X_tr.shape}  labels: {np.bincount(y_tr) if len(y_tr) else []}")
    print(f"  Val  : {X_va.shape}  labels: {np.bincount(y_va) if len(y_va) else []}")
    print(f"  Test : {X_te.shape}  labels: {np.bincount(y_te) if len(y_te) else []}")

    X_tr = apply_fixed_normalisation(X_tr, FEATURES)
    X_va = apply_fixed_normalisation(X_va, FEATURES)
    X_te = apply_fixed_normalisation(X_te, FEATURES)
    print("* Fixed clinical normalisation applied")

    # ── Asymmetric class weights (both minority classes boosted) ─────────────
    classes = np.unique(y_tr)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
    cw      = dict(zip(classes.tolist(), weights.tolist()))
    if 1 in cw:
        cw[1] = cw[1] * CLASS1_BOOST   # critical
    if 2 in cw:
        cw[2] = cw[2] * CLASS2_BOOST   # emergency
    class_weights = np.array([cw.get(c, 1.0) for c in [0,1,2]], dtype=np.float32)
    print(f"* Class Weights: {class_weights}")

    tr_loader = make_loader(X_tr, y_tr, shuffle=True)
    va_loader = make_loader(X_va, y_va, shuffle=False)
    te_loader = make_loader(X_te, y_te, shuffle=False)

    model = CNNGRU(n_features=len(FEATURES), n_classes=3, dropout=0.4).to(DEVICE)
    print(f"* Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    print("\n--- Starting Training ---")
    model, history, best_val_auroc = fit_model(model, tr_loader, va_loader, class_weights)

    # ── Temperature scaling (fit on val logits) ───────────────────────────────
    print("\n* Calibrating temperature on val set...")
    val_logits, val_labels = collect_logits(model, va_loader)
    temp_scaler = TemperatureScaler().to(DEVICE)
    T = temp_scaler.calibrate(val_logits, val_labels)

    # ── Get calibrated val probabilities for threshold tuning ─────────────────
    plain_criterion = nn.CrossEntropyLoss()
    _, _, _, y_val, proba_val = eval_epoch(model, va_loader, plain_criterion,
                                            temp_scaler=temp_scaler)

    # ── Dual-class threshold tuning on val ───────────────────────────────────
    best_t1, best_t2, sweep_df = tune_dual_thresholds(y_val, proba_val)
    plot_threshold_heatmap(sweep_df, OUTPUT_DIR / "threshold_heatmap.png")

    # ── Final test evaluation ─────────────────────────────────────────────────
    _, _, _, y_te_true, proba_te = eval_epoch(model, te_loader, plain_criterion,
                                               temp_scaler=temp_scaler)

    print("\n── Argmax baseline ──")
    res_argmax = decision_report(y_te_true, proba_te, "TEST (argmax)")

    print("\n── Dual threshold ──")
    res_tuned  = decision_report(y_te_true, proba_te,
                                  f"TEST (t1={best_t1:.2f}, t2={best_t2:.2f})",
                                  t1=best_t1, t2=best_t2)

    print(f"\nFinal Val AUROC:    {best_val_auroc:.4f}")
    print(f"Final Test AUROC:   {res_tuned['auroc']:.4f}")
    print(f"Final Test BAL_ACC: {res_tuned['bal']:.4f}")
    print(f"Temperature T:      {T:.4f}")

    cm = res_tuned["cm"]
    print(f"\nNormal  → [Normal={cm[0,0]}, Critical={cm[0,1]}, Emergency={cm[0,2]}]")
    print(f"Critical→ [Normal={cm[1,0]}, Critical={cm[1,1]}, Emergency={cm[1,2]}]")
    print(f"Emergency→[Normal={cm[2,0]}, Critical={cm[2,1]}, Emergency={cm[2,2]}]")

    plot_training(history, OUTPUT_DIR / "training_curves.png")
    plot_confusion(res_argmax["cm"], OUTPUT_DIR / "confusion_argmax.png",  "Confusion (argmax)")
    plot_confusion(cm,               OUTPUT_DIR / "confusion_tuned.png",   f"Confusion (t1={best_t1:.2f}, t2={best_t2:.2f})")

    joblib.dump(
        {
            "state_dict":        model.state_dict(),
            "temp_scaler_state": temp_scaler.state_dict(),
            "temperature":       T,
            "features":          FEATURES,
            "class_weights":     class_weights,
            "best_t1":           best_t1,
            "best_t2":           best_t2,
            "history":           history,
            "best_val_auroc":    best_val_auroc,
        },
        OUTPUT_DIR / "cnn_gru_v6_model.pkl",
    )
    np.save(OUTPUT_DIR / "feature_cols.npy", np.array(FEATURES))
    print(f"\nArtifacts saved to {OUTPUT_DIR}")
    return model, history, res_tuned, best_t1, best_t2


if __name__ == "__main__":
    model, history, test_result, best_t1, best_t2 = main()

Device: cuda
 CNN-GRU
* Initial Data Shape: (2378857, 99)
* Feature engineering...
